In [1]:
import copy

import anndata
import plotnine as p
import scvi
import torch

/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
/home/zeus/miniconda3/envs/cloudspace/lib/python3.10/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!


In [2]:
brain = anndata.io.read_h5ad('250508.brain_train.GSE212576.h5ad')

In [3]:
brain_model = scvi.model.SCVI.load('250508.brain.model', brain)

INFO     File 250508.brain.model/model.pt already downloaded                                                       


In [4]:
liver_model = scvi.model.SCVI.load('250508.liver.model', brain)

INFO     File 250508.liver.model/model.pt already downloaded                                                       


In [5]:
sd_brain = brain_model.module.state_dict()
sd_liver = liver_model.module.state_dict()

In [6]:
sd_brain

OrderedDict([('px_r',
              tensor([-0.2147, -0.4420,  0.3263,  ..., -2.2619, -0.8618, -0.1484],
                     device='cuda:0')),
             ('z_encoder.encoder.fc_layers.Layer 0.0.weight',
              tensor([[-0.0809,  0.0029, -0.0012,  ..., -0.0026,  0.0051, -0.0203],
                      [-0.1276,  0.0036,  0.0062,  ..., -0.0035, -0.0054, -0.0724],
                      [-0.0273,  0.0045,  0.0037,  ..., -0.0056,  0.0017,  0.0816],
                      ...,
                      [-0.0290,  0.0060, -0.0015,  ...,  0.0018,  0.0037, -0.0350],
                      [ 0.0230, -0.0019,  0.0004,  ...,  0.0014, -0.0038, -0.0571],
                      [ 0.0036,  0.0043, -0.0050,  ...,  0.0041, -0.0034, -0.0433]],
                     device='cuda:0')),
             ('z_encoder.encoder.fc_layers.Layer 0.0.bias',
              tensor([ 3.6368e-04, -9.0142e-04, -5.0697e-03,  4.6557e-03, -2.7666e-04,
                      -9.3161e-04,  4.8714e-03,  4.9809e-03, -2.6576e-03, 

In [7]:
sd_liver

OrderedDict([('px_r',
              tensor([ 0.9742,  0.9004,  0.5179,  ...,  0.2667,  0.2356, -0.8414],
                     device='cuda:0')),
             ('z_encoder.encoder.fc_layers.Layer 0.0.weight',
              tensor([[-0.0540, -0.0011,  0.0014,  ..., -0.0042, -0.0001, -0.0131],
                      [ 0.0874,  0.0041,  0.0045,  ...,  0.0016, -0.0021,  0.0056],
                      [ 0.0438,  0.0039, -0.0056,  ...,  0.0039,  0.0031,  0.0021],
                      ...,
                      [ 0.0479, -0.0044,  0.0036,  ..., -0.0026,  0.0054, -0.0861],
                      [ 0.0915, -0.0051,  0.0066,  ...,  0.0033,  0.0028,  0.0068],
                      [-0.0327,  0.0004, -0.0014,  ...,  0.0002,  0.0010, -0.0387]],
                     device='cuda:0')),
             ('z_encoder.encoder.fc_layers.Layer 0.0.bias',
              tensor([-5.8710e-03, -7.7687e-04, -8.4086e-04, -3.3217e-03,  4.2180e-03,
                       4.9510e-03, -3.5026e-03,  2.3965e-03,  4.6576e-03, 

In [8]:
if sd_brain.keys() != sd_liver.keys():
    missing = sd_brain.keys() ^ sd_liver.keys()
    raise ValueError(f"state_dicts differ: {missing}")
else:
    print('Models aligned')

Models aligned


In [9]:
sd_a = sd_brain
sd_b = sd_liver

alpha = 0.5
device = 'cuda'

merged = {}
for k in sd_a:
    t1, t2 = sd_a[k], sd_b[k]
    if t1.shape != t2.shape:
        raise ValueError(f"shape mismatch for {k}: {t1.shape} vs {t2.shape}")
    merged[k] = (alpha * t1) + ((1.0 - alpha) * t2)

In [10]:
merged

{'px_r': tensor([ 0.3797,  0.2292,  0.4221,  ..., -0.9976, -0.3131, -0.4949],
        device='cuda:0'),
 'z_encoder.encoder.fc_layers.Layer 0.0.weight': tensor([[-0.0675,  0.0009,  0.0001,  ..., -0.0034,  0.0025, -0.0167],
         [-0.0201,  0.0039,  0.0053,  ..., -0.0010, -0.0037, -0.0334],
         [ 0.0082,  0.0042, -0.0009,  ..., -0.0008,  0.0024,  0.0418],
         ...,
         [ 0.0095,  0.0008,  0.0010,  ..., -0.0004,  0.0046, -0.0606],
         [ 0.0573, -0.0035,  0.0035,  ...,  0.0024, -0.0005, -0.0252],
         [-0.0146,  0.0023, -0.0032,  ...,  0.0021, -0.0012, -0.0410]],
        device='cuda:0'),
 'z_encoder.encoder.fc_layers.Layer 0.0.bias': tensor([-2.7536e-03, -8.3914e-04, -2.9553e-03,  6.6701e-04,  1.9707e-03,
          2.0097e-03,  6.8440e-04,  3.6887e-03,  1.0000e-03, -1.5358e-03,
         -2.0645e-04,  6.0516e-03, -3.0089e-05,  1.5914e-03,  1.4367e-03,
          4.3800e-03, -4.6241e-04, -1.4326e-04, -1.1885e-03, -7.3904e-04,
          9.7948e-04,  6.0348e-04,  2.2

In [11]:
brain_model.module.load_state_dict(merged)

<All keys matched successfully>

In [12]:
brain_model.save('250517.linear.merge.model', overwrite = True)

In [18]:
brain_model = scvi.model.SCVI.load('250508.brain.model', brain)

INFO     File 250508.brain.model/model.pt already downloaded                                                       


In [19]:
def _nuslerp_vec(a: torch.Tensor, b: torch.Tensor, t: float, eps=1e-8):
    """NuSLERP on two 1-D vectors (PyTorch tensors on the same device)."""
    na, nb = a.norm(), b.norm()
    if na < eps or nb < eps:               # degenerate: fallback to lerp
        return (1.0 - t) * a + t * b

    a_hat, b_hat = a / na, b / nb          # 1) normalise
    dot = torch.clamp((a_hat * b_hat).sum(), -1.0, 1.0)
    omega = torch.acos(dot)                # angle between the two
    if omega.abs() < 1e-6:                 # almost identical → lerp
        unit = (1.0 - t) * a_hat + t * b_hat
    else:                                  # 2) spherical lerp
        unit = (torch.sin((1.0 - t) * omega) * a_hat +
                torch.sin(t * omega)       * b_hat) / torch.sin(omega)
    scale = (1.0 - t) * na + t * nb        # 3) renormalise
    return unit * scale

def _nuslerp_tensor(tA: torch.Tensor, tB: torch.Tensor, t: float,
                    row_wise=False):
    """Apply NuSLERP to an N-D tensor."""
    if row_wise and tA.ndim >= 2:          # run per-row to keep channels aligned
        out = torch.empty_like(tA)
        for i in range(tA.shape[0]):
            out[i] = _nuslerp_tensor(tA[i], tB[i], t, row_wise=False)
        return out
    else:                                  # flatten → NuSLERP → reshape
        merged = _nuslerp_vec(tA.flatten(), tB.flatten(), t)
        return merged.view_as(tA)

In [20]:
row_wise = False

nuslerp_merged = {}
for k in sd_a:
    ta, tb = sd_a[k], sd_b[k]
        # --- skip non-floating tensors ------------------------------------
    if not (ta.is_floating_point() or ta.is_complex()):
        merged[k] = ta.clone()          # just take A’s value (fine for ints)
        continue

    if ta.shape != tb.shape:
        raise ValueError(f"{k}: shape mismatch {ta.shape} vs {tb.shape}")
    with torch.no_grad():
        nuslerp_merged[k] = _nuslerp_tensor(ta.to(device), tb.to(device),
                                    alpha, row_wise=row_wise)

In [21]:
brain_model.module.load_state_dict(merged)

<All keys matched successfully>

In [22]:
brain_model.save('250517.nuslerp.merge.model', overwrite = True)